# 01 FTH

            First notebook in the workflow. It loads the raw holograms, centers them,
            optionally applies the Ewald projection, defines an ROI, creates a simple
            radially symmetric Butterworth smooth beamstop mask, reconstructs the FTH
            image, and saves the nested `data` dictionary to HDF5.

            Output: `processed/Logs/data_recon_ImId_<im_id>_<user>.hdf5`.

In [ ]:
import os, sys
from pathlib import Path
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector


def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())


BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf

wf = reload(wf)  # Refresh helpers when rerunning in an existing kernel.
from data_loading import SextantsNexusLoader, image_ids, load_average
from image_preprocessing import fit_dark_frame, fit_horizontal_band, load_detector_masks

try:
    import cupy as cp
    import cupyx as cpx
    import CCI_core_cupy as cci
    import Phase_Retrieval as PhR

    GPU = True
    print("GPU available")
except Exception:
    import CCI_core as cci

    PhR = None
    GPU = False
    print("GPU unavailable")

%matplotlib qt
try:
    %load_ext jupyter_black
except Exception:
    pass

## Folders and user

In [ ]:
BASEFOLDER = Path(find_basefolder())
RAW_FOLDER = Path("/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/")
RAW_DATA_KIND = "sextants_nexus"
DATAFOLDER = RAW_FOLDER
USER = "rb"

folder_general = helper.create_folder(BASEFOLDER / "processed")
folder_logs = helper.create_folder(Path(folder_general) / "Logs")
print("Raw folder:", RAW_FOLDER)
print("Output folder:", folder_general)


## Define raw images

In [ ]:
# Current acquisition parameters from the newer working notebook.
# A single ID or a list is accepted; lists are averaged as floating point.
PLUS_IMAGE_IDS = [613]
MINUS_IMAGE_IDS = [614]
PLUS_DARK_IDS = 604
MINUS_DARK_IDS = 604

# The fixed detector mask is always processed/mask_pixels/mask_detector.png.
# These IDs select mask_beamstop_<ID>.png in the same folder.
PLUS_BEAMSTOP_MASK_ID = 95
MINUS_BEAMSTOP_MASK_ID = 95
PLUS_STITCHED_FILE = None
MINUS_STITCHED_FILE = None

POSITIVE_LABEL = "+"
REFERENCE_LABEL = "-"
CROP = None
PROJECT_EWALD_SPHERE = False
EWALD_METHOD = "cubic"
HERALDO = False

MASK_FOLDER = BASEFOLDER / "processed" / "mask_pixels"
MASK_DETECTOR_FILE = MASK_FOLDER / "mask_detector.png"
PLUS_BEAMSTOP_MASK_FILE = MASK_FOLDER / f"mask_beamstop_{PLUS_BEAMSTOP_MASK_ID}.png"
MINUS_BEAMSTOP_MASK_FILE = MASK_FOLDER / f"mask_beamstop_{MINUS_BEAMSTOP_MASK_ID}.png"

print("Fixed detector mask:", MASK_DETECTOR_FILE)
print("Plus beamstop mask:", PLUS_BEAMSTOP_MASK_FILE)
print("Minus beamstop mask:", MINUS_BEAMSTOP_MASK_FILE)

# This nested object is generated only for the established HDF5 format.
hologram_inputs = {
    "+": {
        "id": PLUS_IMAGE_IDS,
        "dark_id": PLUS_DARK_IDS,
        "beamstop_mask_id": PLUS_BEAMSTOP_MASK_ID,
        "stitched_file": PLUS_STITCHED_FILE,
    },
    "-": {
        "id": MINUS_IMAGE_IDS,
        "dark_id": MINUS_DARK_IDS,
        "beamstop_mask_id": MINUS_BEAMSTOP_MASK_ID,
        "stitched_file": MINUS_STITCHED_FILE,
    },
}
positive_label = POSITIVE_LABEL
reference_label = REFERENCE_LABEL
crop = CROP
project_ewalds_sphere = PROJECT_EWALD_SPHERE
ewald_method = EWALD_METHOD
heraldo = HERALDO


## Derived paths and experimental setup

In [ ]:
# Everything in this cell is derived from the folders and raw-image cells above.
raw_loader = (
    SextantsNexusLoader(RAW_FOLDER) if RAW_DATA_KIND == "sextants_nexus" else None
)
if raw_loader is None:
    raise ValueError(
        "Automatic experimental metadata currently requires RAW_DATA_KIND='sextants_nexus'"
    )

im_ids = image_ids(hologram_inputs[positive_label]["id"])
im_id = int(im_ids[0])  # First ID is used only for metadata and filenames.
positive_file = raw_loader.path_for(im_id)
ccd_dist_source = f"/scan_{im_id:04d}/scan_data/data_03"
with h5py.File(positive_file, "r") as handle:
    if ccd_dist_source not in handle:
        raise KeyError(f"Missing {ccd_dist_source} in {positive_file}")
    ccd_dataset = handle[ccd_dist_source]
    ccd_dist_mm = float(np.asarray(ccd_dataset[()]).squeeze())
ccd_dist_m = (700.0 - ccd_dist_mm) / 1000.0

setup_frame = raw_loader.load(im_id)
if "energy_eV" not in setup_frame.metadata:
    raise KeyError(f"No photon energy found in positive image {im_id} NeXus metadata")

energy = float(setup_frame.metadata["energy_eV"])
experimental_setup = {
    "ccd_dist": ccd_dist_m,
    "ccd_dist_source": ccd_dist_source,
    "ccd_dist_image_id": im_id,
    "px_size": 11.0e-6,
    "binning": 1,
    "oversaturation": 60e3,
    "energy": energy,
    "lambda": helper.photon_energy_wavelength(energy, input_unit="eV"),
}
detector = Detector(
    experimental_setup["binning"] * experimental_setup["px_size"],
    experimental_setup["binning"] * experimental_setup["px_size"],
)
mnemonics = loading.mnemonics
DATA_H5 = join(folder_logs, f"data_recon_ImId_{im_id:04d}_{USER}.hdf5")

data = {
    "workflow": "FTH_phase_retrieval_4_notebook_sequence",
    "user": USER,
    "data_file": DATA_H5,
    "experimental_setup": experimental_setup,
    "holo": hologram_inputs,
    "hologram_labels": list(hologram_inputs.keys()),
    "positive_label": positive_label,
    "reference_label": reference_label,
    "heraldo": heraldo,
    "crop": crop,
}

print("Positive images:", im_ids)
print("Output HDF5:", DATA_H5)
print(
    "CCD distance:",
    experimental_setup["ccd_dist"],
    "m from",
    experimental_setup["ccd_dist_source"],
)
print("Pixel size:", experimental_setup["px_size"], "m")
print("Energy:", experimental_setup["energy"], "eV")

## Load raw data

In [ ]:
# Dark normalization is always fitted in this corner and excludes detector defects.
DARK_FIT_ROWS = slice(0, 600)
DARK_FIT_COLUMNS = slice(0, 200)
DARK_FIT_PERCENTILE = 100
DARK_FIT_STRIDE = 1


def load_raw_or_stitched(image_ids, dark_ids, beamstop_mask_id, stitched_file, label):
    if stitched_file is not None:
        stitched_path = Path(stitched_file)
        if not stitched_path.is_absolute():
            stitched_path = BASEFOLDER / stitched_path
        with np.load(stitched_path, allow_pickle=False) as saved:
            image = np.asarray(saved["image"], dtype=float)
            mask_pixel = np.asarray(saved["mask_pixel"], dtype=np.uint8)
            mask_detector = np.asarray(saved["mask_detector"], dtype=np.uint8)
            mask_beamstop = np.asarray(saved["mask_beamstop"], dtype=np.uint8)
            metadata = {"energy_eV": float(saved["energy_eV"])}
            exposure = float(saved["reference_exposure"])
        print(f"{label}: stitched input {stitched_path}")
        return image, mask_detector, mask_beamstop, mask_pixel, metadata, exposure, str(stitched_path), None

    frame = load_average(raw_loader, image_ids)
    image = np.asarray(frame.image, dtype=float)
    mask_detector, mask_beamstop, mask_pixel = load_detector_masks(
        MASK_FOLDER, beamstop_mask_id, image.shape
    )

    # Fit average_image = scale * average_dark + offset in the selected corner.
    dark = np.asarray(load_average(raw_loader, dark_ids).image, dtype=float)
    image, dark_scale, dark_offset, dark_values, fitted_pixels = fit_dark_frame(
        image,
        dark,
        DARK_FIT_ROWS,
        DARK_FIT_COLUMNS,
        percentile=DARK_FIT_PERCENTILE,
        stride=DARK_FIT_STRIDE,
        mask=mask_detector,
    )
    image_values = frame.image[DARK_FIT_ROWS, DARK_FIT_COLUMNS][
        ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
    ].ravel()
    fig, axis = plt.subplots(figsize=(6, 5))
    axis.scatter(dark_values[fitted_pixels], image_values[fitted_pixels], s=3, alpha=0.15)
    fit_x = np.linspace(dark_values[fitted_pixels].min(), dark_values[fitted_pixels].max(), 200)
    axis.plot(fit_x, dark_scale * fit_x + dark_offset, color="red", linewidth=2)
    axis.set_title(f"{label}: fitted dark normalization")
    axis.set_xlabel("Dark intensity")
    axis.set_ylabel("Image intensity")
    axis.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
    print(f"{label}: dark scale={dark_scale:.6g}, offset={dark_offset:.6g}")
    return image, mask_detector, mask_beamstop, mask_pixel, dict(frame.metadata), frame.exposure, str(frame.source), (dark_scale, dark_offset)


plus_loaded = load_raw_or_stitched(
    PLUS_IMAGE_IDS, PLUS_DARK_IDS, PLUS_BEAMSTOP_MASK_ID, PLUS_STITCHED_FILE, "+"
)
minus_loaded = load_raw_or_stitched(
    MINUS_IMAGE_IDS, MINUS_DARK_IDS, MINUS_BEAMSTOP_MASK_ID, MINUS_STITCHED_FILE, "-"
)

# Simple NumPy names are available immediately after loading.
pos_raw = np.asarray(plus_loaded[0], dtype=float)
neg_raw = np.asarray(minus_loaded[0], dtype=float)
mask_detector_pos = np.asarray(plus_loaded[1], dtype=np.uint8)
mask_detector_neg = np.asarray(minus_loaded[1], dtype=np.uint8)
mask_beamstop_pos = np.asarray(plus_loaded[2], dtype=np.uint8)
mask_beamstop_neg = np.asarray(minus_loaded[2], dtype=np.uint8)
mask_pixel_pos = np.asarray(plus_loaded[3], dtype=np.uint8)
mask_pixel_neg = np.asarray(minus_loaded[3], dtype=np.uint8)

for label, loaded in (("+", plus_loaded), ("-", minus_loaded)):
    image, mask_detector, mask_beamstop, mask_pixel_raw, metadata, exposure, source_name, dark_fit = loaded
    state = data["holo"][label]
    state["image"] = image
    state["mask_detector_raw"] = mask_detector
    state["mask_beamstop_raw"] = mask_beamstop
    state["mask_pixel_raw"] = mask_pixel_raw
    state["raw_metadata"] = metadata
    state["exposure"] = exposure
    state["spe_name"] = source_name
    state["dark_fit"] = dark_fit

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, label in enumerate(("+", "-")):
    state = data["holo"][label]
    vmin, vmax = wf.finite_percentile_limits(state["image"], (1, 99.9))
    axes[row, 0].imshow(state["image"], cmap="viridis", vmin=vmin, vmax=vmax)
    axes[row, 0].set_title(f"{label}: dark-corrected image")
    axes[row, 1].imshow(state["mask_detector_raw"], cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("fixed mask_detector")
    axes[row, 2].imshow(state["mask_beamstop_raw"], cmap="gray", vmin=0, vmax=1)
    axes[row, 2].set_title("mask_beamstop")
    axes[row, 3].imshow(state["mask_pixel_raw"], cmap="gray", vmin=0, vmax=1)
    axes[row, 3].set_title("combined mask_pixel")
    for axis in axes[row]:
        axis.set_axis_off()
plt.tight_layout()
plt.show()


## Choose center

In [ ]:
# Starting center retained from the newer working notebook.
pol = reference_label
c0, c1 = [997, 1040]
ic = interactive.InteractiveCenter(neg_raw, c0=c0, c1=c1)

In [ ]:
# Get center positions from the widget.
center = [ic.c0, ic.c1]
print("Center:", center)
data["center"] = center
data = wf.define_centered_holograms(
    data,
    cci,
    PhR=PhR,
    project_ewalds_sphere=project_ewalds_sphere,
    ewald_method=ewald_method,
)
for label, state in data["holo"].items():
    state["mask_detector_c"] = (
        wf.center_image(state["mask_detector_raw"], data["center"], cci) > 0.5
    ).astype(np.uint8)
    state["mask_beamstop_c"] = (
        wf.center_image(state["mask_beamstop_raw"], data["center"], cci) > 0.5
    ).astype(np.uint8)
    state["mask_pixel_c"] = np.clip(
        state["mask_detector_c"] + state["mask_beamstop_c"], 0, 1
    ).astype(np.uint8)
# A difference hologram is valid only where both input images are valid.
mask_pixel = np.maximum(
    data["holo"][positive_label]["mask_pixel_c"],
    data["holo"][reference_label]["mask_pixel_c"],
).astype(np.uint8)
data["mask_pixel_raw_by_label"] = {
    label: state["mask_pixel_raw"] for label, state in data["holo"].items()
}
data["mask_pixel"] = mask_pixel
plot_labels = list(data["holo"].keys())
print("Plot order:", plot_labels)
cimshow(np.stack([data["holo"][label]["image_c"] for label in plot_labels]))

## Smooth Butterworth mask and FTH reconstruction

In [ ]:
# Optional horizontal-band correction. Parameters stay beside the operation.
CORRECT_HORIZONTAL_BAND = True
BAND_EDGE_COLUMNS = 20
BAND_SKIPPED_EDGE_COLUMNS = 0
BAND_POLYNOMIAL_ORDER = 2
BAND_CENTER = 1024
BAND_WIDTH = 80
BAND_EDGE = 11
SUBTRACT_POLYNOMIAL_BACKGROUND = False


In [ ]:
# FTH masking parameters are here because the masks are constructed below.
# mask_pixel remains binary for phase retrieval; these smooth masks are FTH-only.
MASK_PIXEL_DILATION = 3
MASK_PIXEL_BLUR_SIGMA = 3.0
BUTTERWORTH_RADIUS = 35
BUTTERWORTH_ORDER = 4
prop_dist = 0
phase = 0

shape = data["holo"][positive_label]["image_c"].shape
if mask_pixel.shape != shape:
    raise ValueError(f"Centered mask shape {mask_pixel.shape} != hologram shape {shape}")

# 1) Slightly enlarge mask_pixel, then blur its edge.
mask_pixel_fth = wf.smooth_binary_mask(
    mask_pixel.astype(float),
    MASK_PIXEL_DILATION,
    MASK_PIXEL_BLUR_SIGMA,
)

# 2) Apply a smooth Butterworth stop around the Fourier origin.
mask_beamstop_smooth = wf.butterworth_disk_mask(
    shape,
    BUTTERWORTH_RADIUS,
    BUTTERWORTH_ORDER,
)

# Both arrays above are exclusion masks: 0 is retained and 1 is removed.
mask_multiplier = (1 - mask_pixel_fth) * (1 - mask_beamstop_smooth)

mask_pixel_fth_recipe = {
    "type": "binary_dilation_gaussian",
    "dilation_pixels": MASK_PIXEL_DILATION,
    "sigma": MASK_PIXEL_BLUR_SIGMA,
}
mask_beamstop_smooth_recipe = {
    "type": "radial_butterworth_disk",
    "radius": BUTTERWORTH_RADIUS,
    "order": BUTTERWORTH_ORDER,
}

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for axis, shown_mask, title in zip(
    axes,
    (mask_pixel, mask_pixel_fth, mask_beamstop_smooth, mask_multiplier),
    ("binary mask_pixel", "dilated + blurred mask_pixel",
     "Butterworth exclusion mask", "final FTH transmission"),
):
    image = axis.imshow(shown_mask, cmap="gray", vmin=0, vmax=1)
    axis.set_title(title)
    axis.set_axis_off()
    fig.colorbar(image, ax=axis, fraction=0.046)
plt.tight_layout()
plt.show()

pos = np.asarray(data["holo"][positive_label]["image_c"], dtype=float)
neg = np.asarray(data["holo"][reference_label]["image_c"], dtype=float)
if CORRECT_HORIZONTAL_BAND:
    band_fits = []
    for label, image in ((positive_label, pos), (reference_label, neg)):
        band_fit = fit_horizontal_band(
            image,
            edge_columns=BAND_EDGE_COLUMNS,
            skipped_edge_columns=BAND_SKIPPED_EDGE_COLUMNS,
            polynomial_order=BAND_POLYNOMIAL_ORDER,
            band_center=BAND_CENTER,
            band_width=BAND_WIDTH,
            band_edge=BAND_EDGE,
            mask=mask_pixel,
        )
        band_fits.append(band_fit)
        fig, axis = plt.subplots(figsize=(10, 4))
        axis.plot(band_fit.rows, band_fit.measured_profile, label="measured edge profile")
        axis.plot(band_fit.rows, band_fit.fitted_profile, linewidth=2, label="complete fit")
        axis.plot(band_fit.rows, band_fit.polynomial_profile, "--", label=f"polynomial order {BAND_POLYNOMIAL_ORDER}")
        axis.fill_between(band_fit.rows, band_fit.polynomial_profile, band_fit.fitted_profile, alpha=0.25, label="negative band")
        axis.set_title(f"{label}: horizontal-band fit")
        axis.set_xlabel("Detector row [pixel]")
        axis.set_ylabel("Intensity")
        axis.legend()
        axis.grid(alpha=0.2)
        plt.tight_layout()
        plt.show()
    pos = pos - band_fits[0].band_image
    neg = neg - band_fits[1].band_image
    if SUBTRACT_POLYNOMIAL_BACKGROUND:
        pos = pos - band_fits[0].polynomial_image
        neg = neg - band_fits[1].polynomial_image

# Fit polarization normalization only outside mask_detector + mask_beamstop.
POLARIZATION_FIT_ROWS = slice(500, -500)
POLARIZATION_FIT_COLUMNS = slice(500, -500)
POLARIZATION_FIT_PERCENTILES = (2, 98)

fit_region = np.zeros(pos.shape, dtype=bool)
fit_region[POLARIZATION_FIT_ROWS, POLARIZATION_FIT_COLUMNS] = True
fit_pixels = (
    (mask_pixel == 0)
    & fit_region
    & np.isfinite(pos)
    & np.isfinite(neg)
)
low, high = np.percentile(pos[fit_pixels], POLARIZATION_FIT_PERCENTILES)
fit_pixels &= (pos >= low) & (pos <= high)
factor, offset = cci.dyn_factor(
    pos[fit_pixels],
    neg[fit_pixels],
    method="correlation",
    verbose=False,
    plot=False,
)
pos_normalized = pos / factor

plot_step = max(1, fit_pixels.sum() // 7000)
fit_x_values = pos[fit_pixels][::plot_step]
fit_y_values = neg[fit_pixels][::plot_step]
fit_x = np.linspace(fit_x_values.min(), fit_x_values.max(), 300)
fig, axis = plt.subplots(figsize=(6, 5))
axis.scatter(fit_x_values, fit_y_values, s=3, alpha=0.15, label="valid fit pixels")
axis.plot(fit_x, fit_x / factor - offset, color="red", linewidth=2,
          label=f"fit: x/{factor:.5g} - {offset:.5g}")
axis.set_title("Polarization normalization outside mask_pixel")
axis.set_xlabel(f"{positive_label} intensity")
axis.set_ylabel(f"{reference_label} intensity")
axis.legend()
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()
holo_unmasked = pos_normalized - neg - offset
holo_masked = holo_unmasked * mask_multiplier
recon_unmasked = wf.fth_reconstruct(
    holo_unmasked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)
recon_masked = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth, prop_dist=prop_dist, phase=phase
)

data["mask_pixel"] = mask_pixel
data["mask_beamstop_smooth_recipe"] = mask_beamstop_smooth_recipe
data["mask_pixel_fth_recipe"] = mask_pixel_fth_recipe
data["factor"] = factor
data["offset"] = offset
data["holo"][positive_label]["image_c_norm"] = wf.normalize_image(pos_normalized)
data["holo"][reference_label]["image_c_norm"] = wf.normalize_image(neg)
data["focus_fth"] = {
    "prop_dist": prop_dist,
    "prop_dist_unit": "um",
    "phase": phase,
    "dx": 0,
    "dy": 0,
    "roi": None,
    "operation": "-",
}
data["fth_recipe"] = {
    "positive_label": positive_label,
    "reference_label": reference_label,
    "center": data["center"],
    "mask_beamstop_smooth_recipe": mask_beamstop_smooth_recipe,
    "mask_pixel_fth_recipe": mask_pixel_fth_recipe,
    "project_ewalds_sphere": data["project_ewalds_sphere"],
    "ewald_method": ewald_method,
    "contrast": "positive / factor - reference - offset",
}

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
tmp = holo_masked
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (0.1, 99.9))
ax[0].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[0].add_patch(
    plt.Circle(
        (shape[1] / 2, shape[0] / 2),
        butterworth_radius,
        fill=False,
        edgecolor="red",
        linewidth=1.5,
    )
)
ax[0].set_title("mask * hologram")

tmp = np.real(recon_unmasked)
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[1].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[1].set_title("FTH before masking")
tmp = np.real(recon_masked)
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[2].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[2].set_title("FTH after masking")
for axis in ax:
    axis.set_axis_off()
plt.show()

In [ ]:
interactive.cimshow(pos, vmin=0, vmax=1000)

In [ ]:
fig, ax = plt.subplots()
ax.imshow(pos, vmin=-100, vmax=100)

## Define ROI from masked FTH reconstruction

In [ ]:
# Preview the masked FTH reconstruction. The next cell applies the calibrated fixed ROI.
# fig, ax = cimshow(np.real(recon_masked))

In [ ]:
# Fixed ROI calibrated with the image-13 center above.
roi = [597, 785, 526, 719]
roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]
print("ROI:", roi)

## Focus FTH reconstruction

In [ ]:
# Use the focusCDI widget to tune propagation distance and phase.
# The selected values are committed to the HDF5 data dictionary in the next cell.
# Manual starting values: propagation is in micrometres, phase in radians,
# and dx/dy are sub-pixel shifts. Edit these before creating the sliders.
prop_dist, phase, dx, dy = -5, -0.27 - np.pi, 0, 0.0
focus_operation = "-"
focus_sliders = rec.focusCDI(
    holo_masked,
    np.zeros_like(holo_masked),
    roi_s,
    mask=1,
    phase=phase,
    prop_dist=prop_dist,
    dx=dx,
    dy=dy,
    experimental_setup=data["experimental_setup"],
    operation=focus_operation,
    max_prop_dist=30,
    scale=(2, 98),
)
slider_prop, slider_phase, slider_dx, slider_dy = focus_sliders[:4]

In [ ]:
# Commit selected focus values and recompute the saved FTH reconstruction.
prop_dist = slider_prop.value
phase = slider_phase.value
dx = slider_dx.value
dy = slider_dy.value

focus_fth = {
    "prop_dist": prop_dist,
    "prop_dist_unit": "um",
    "phase": phase,
    "dx": dx,
    "dy": dy,
    "roi": roi,
    "operation": focus_operation,
}
data["focus_fth"] = focus_fth

recon_unmasked = wf.fth_reconstruct(
    holo_unmasked,
    experimental_setup,
    fth,
    prop_dist=prop_dist,
    phase=phase,
    dx=dx,
    dy=dy,
)
recon_masked = wf.fth_reconstruct(
    holo_masked, experimental_setup, fth, prop_dist=prop_dist, phase=phase, dx=dx, dy=dy
)

data["recon"] = recon_masked[roi_s]
data["recon_description"] = (
    "Complex FTH reconstruction after propagation, phase shift, Butterworth masking, and ROI crop."
)

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
tmp = np.real(recon_unmasked)[roi_s]
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[0].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[0].set_title("Focused FTH before masking")
tmp = np.real(recon_masked)[roi_s]
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax[1].imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax[1].set_title("Focused FTH after masking")
for axis in ax:
    axis.set_axis_off()
plt.show()

print("prop_dist:", prop_dist)
print("phase:", phase)

im_ids = image_ids(data["holo"][positive_label]["id"])
im_id = int(im_ids[0])
png_title = f"Focused FTH after masking - im_id {im_id}"
png_name = join(folder_general, f"FTH_recon_ImId_{int(im_id):04d}_{USER}.png")
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(data["recon"])
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(png_title)
ax.set_axis_off()
plt.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.show()

data["fth_png"] = png_name
print("Saved PNG:", png_name)

## Save

In [ ]:
# Always write the display PNG in the same final cell as the HDF5 result.
im_ids = image_ids(data["holo"][positive_label]["id"])
im_id = int(im_ids[0])
png_name = join(folder_general, f"FTH_recon_ImId_{int(im_id):04d}_{USER}.png")
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(data["recon"])
vmin, vmax = np.percentile(tmp[np.isfinite(tmp)], (1, 99))
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(f"Focused FTH after masking - im_id {im_id}")
ax.set_axis_off()
fig.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.close(fig)
data["fth_png"] = png_name
written = wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Saved HDF5:", written)
print("Saved PNG:", png_name)

In [ ]:
# Workflow summary
_summary_data = data if "data" in globals() and isinstance(data, dict) else {}
_summary_h5 = globals().get("DATA_H5", _summary_data.get("data_file", "n/a"))
_summary_holo = _summary_data.get("holo", {})
_summary_pos = _summary_data.get(
    "positive_label", globals().get("positive_label", None)
)
_summary_ref = _summary_data.get(
    "reference_label", globals().get("reference_label", None)
)
_summary_im = _summary_holo.get(_summary_pos, {}).get(
    "id", globals().get("im_id", "n/a")
)
_summary_topo = _summary_holo.get(_summary_ref, {}).get(
    "id", globals().get("topo_id", "n/a")
)
print("im_ids:", _summary_im)
print("topo_ids:", _summary_topo)
print("dark_ids (+):", _summary_holo.get(_summary_pos, {}).get("dark_id"))
print("dark_ids (-):", _summary_holo.get(_summary_ref, {}).get("dark_id"))
print("HDF5:", _summary_h5)

In [ ]:
# Acquisition ID summary
_id_holo = data.get("holo", {})
print("im_ids:", {label: state.get("id") for label, state in _id_holo.items()})
print("dark_ids:", {label: state.get("dark_id") for label, state in _id_holo.items()})

## Simple names for interactive inspection

Use these names directly when making an extra plot. Holograms use `viridis`; `reconstruction_fth` is a real-space image and may be shown in gray.


In [ ]:
# Conventional short NumPy names used throughout older FTH notebooks.
pos = np.asarray(pos, dtype=float)
neg = np.asarray(neg, dtype=float)
holo = np.asarray(holo_masked, dtype=float)
recon = np.asarray(data["recon"])


def show_image(image, title="image", cmap="viridis", percentiles=(1, 99.9)):
    image = np.asarray(image)
    values = np.real(image) if np.iscomplexobj(image) else image.astype(float)
    finite = values[np.isfinite(values)]
    vmin, vmax = np.percentile(finite, percentiles)
    fig, axis = plt.subplots(figsize=(6, 5))
    shown = axis.imshow(values, cmap=cmap, vmin=vmin, vmax=vmax)
    axis.set_title(title)
    axis.set_axis_off()
    fig.colorbar(shown, ax=axis)
    plt.tight_layout()
    plt.show()
    return fig, axis


print("Easy NumPy names: pos_raw, neg_raw, pos, neg, mask_pixel, holo, recon")
print("Example: show_image(pos, 'positive image')")
print("Example: show_image(holo, 'FTH hologram')")
print("Example: show_image(recon, 'FTH reconstruction', cmap='gray')")
